# 02 — Head-only fine-tuning

Ноутбук воспроизводит только head-only эксперимент (`lm_head`) для Gazeta.
Сводные метрики, сравнение подходов и общий разбор проблем Kaggle описаны в `README.md`.

Порядок запуска:
1) Выполнить install-ячейку
2) Перезапустить сессию
3) Запустить оставшиеся ячейки сверху вниз

Артефакт: `/kaggle/working/02_head_only_rouge.json`

In [1]:

!pip -q install -U "transformers==4.46.1" "accelerate>=0.34.0" "datasets>=2.18.0" "trl>=0.9.6" "peft>=0.12.0" "rouge-score" "sentencepiece"
# IMPORTANT: pin protobuf to avoid: AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
!pip -q install -U "protobuf==4.25.3"
print("Installed. NOW do: Session → Restart Session, then run remaining cells.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 70.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.0/348.0 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 84.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 5.5 MB/s eta 0:00:00
ER

## Imports + helpers

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # force single GPU (avoid DataParallel replicas/oom)
os.environ["TOKENIZERS_PARALLELISM"] = "false"


import os, json, random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

from rouge_score import rouge_scorer, scoring

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

WORKDIR = "/kaggle/working"
os.makedirs(WORKDIR, exist_ok=True)

def find_gazeta_jsonl():
    root = "/kaggle/input"
    paths = []
    if not os.path.exists(root):
        raise FileNotFoundError("No /kaggle/input found. Attach the Gazeta Summaries dataset in the right panel.")
    for dirpath, _, filenames in os.walk(root):
        for fn in filenames:
            low = fn.lower()
            if low.endswith(".jsonl") and "gazeta" in low:
                paths.append(os.path.join(dirpath, fn))
    return sorted(paths)

def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)

def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    low2orig = {c.lower(): c for c in df.columns}
    text_col = low2orig.get("text") or low2orig.get("article") or low2orig.get("document") or low2orig.get("source")
    sum_col  = low2orig.get("summary") or low2orig.get("target") or low2orig.get("abstract")
    if text_col is None or sum_col is None:
        raise ValueError(f"Can't infer text/summary columns. Columns: {list(df.columns)}")
    out = df.rename(columns={text_col: "text", sum_col: "summary"})[["text", "summary"]].copy()
    out["text"] = out["text"].astype(str)
    out["summary"] = out["summary"].astype(str)
    return out

def load_gazeta_splits():
    paths = find_gazeta_jsonl()
    if not paths:
        raise FileNotFoundError("No gazeta*.jsonl found under /kaggle/input. Make sure the dataset is attached.")
    def pick(key):
        cands = [p for p in paths if key in os.path.basename(p).lower()]
        if not cands:
            raise FileNotFoundError(f"Can't find *{key}*.jsonl in: {paths}")
        return cands[0]
    train_path = pick("train")
    val_path   = pick("val")
    test_path  = pick("test")

    train_df = normalize_columns(read_jsonl(train_path))
    val_df   = normalize_columns(read_jsonl(val_path))
    test_df  = normalize_columns(read_jsonl(test_path))

    print("Loaded:",
          os.path.basename(train_path), train_df.shape,
          "|", os.path.basename(val_path), val_df.shape,
          "|", os.path.basename(test_path), test_df.shape)
    return train_df, val_df, test_df

def rouge_f1(preds, refs):
    scorer = rouge_scorer.RougeScorer(["rouge1","rouge2","rougeL"], use_stemmer=True)
    agg = scoring.BootstrapAggregator()
    for p, r in zip(preds, refs):
        agg.add_scores(scorer.score(r, p))
    res = agg.aggregate()
    return {
        "rouge1": float(res["rouge1"].mid.fmeasure),
        "rouge2": float(res["rouge2"].mid.fmeasure),
        "rougeL": float(res["rougeL"].mid.fmeasure),
    }


2026-03-02 20:25:14.815300: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772483115.227158      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772483115.337172      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772483116.437237      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772483116.437281      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772483116.437284      24 computation_placer.cc:177] computation placer alr

## Load data (small subset for stable runtime)

In [3]:

train_df, val_df, test_df = load_gazeta_splits()

TRAIN_N = min(2000, len(train_df))
VAL_N   = min(400, len(val_df))
TEST_N  = min(400, len(test_df))

train_df_s = train_df.sample(n=TRAIN_N, random_state=SEED).reset_index(drop=True)
val_df_s   = val_df.sample(n=VAL_N, random_state=SEED).reset_index(drop=True)
test_df_s  = test_df.sample(n=TEST_N, random_state=SEED).reset_index(drop=True)

train_df_s.head(2)


Loaded: gazeta_train.jsonl (60964, 2) | gazeta_val.jsonl (6369, 2) | gazeta_test.jsonl (6793, 2)


,text,summary
0,"«Нас ждет первая серьезная игра в сезоне, — ск...",Главный тренер «Анжи» Гус Хиддинк накануне мат...
1,"Орбитальная группировка спутников России, по п...",Комиссия по выяснению причин срыва обновления ...


## Tokenization

In [4]:

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

MAX_LEN = 384

def build_prompt(article: str, summary: str | None = None):
    inst = "Сделай краткое резюме новости."
    if summary is None:
        return f"[INST] {inst}\n\nТекст:\n{article}\n\nОтвет:\n[/INST]"
    return f"[INST] {inst}\n\nТекст:\n{article}\n\nОтвет:\n{summary} [/INST]"

def tokenize_batch(batch):
    prompts = [build_prompt(a, s) for a, s in zip(batch["text"], batch["summary"])]
    enc = tokenizer(prompts, truncation=True, max_length=MAX_LEN, padding="max_length")
    enc["labels"] = enc["input_ids"].copy()
    return enc

# IMPORTANT: remove_columns must be a plain list (NOT pandas Index) to avoid numpy truth-value error
train_ds = Dataset.from_pandas(train_df_s).map(
    tokenize_batch, batched=True, remove_columns=list(train_df_s.columns)
)
val_ds = Dataset.from_pandas(val_df_s).map(
    tokenize_batch, batched=True, remove_columns=list(val_df_s.columns)
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

## Model: untie + freeze all except lm_head

In [5]:

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map={"": 0},
)

# Memory savers
model.config.use_cache = False
model.gradient_checkpointing_enable()

# Untie lm_head (Qwen usually ties it with embeddings)
old_head = model.get_output_embeddings()
hidden = old_head.weight.shape[1]
vocab = old_head.weight.shape[0]

new_head = torch.nn.Linear(hidden, vocab, bias=False, dtype=old_head.weight.dtype)
new_head = new_head.to(old_head.weight.device)
with torch.no_grad():
    new_head.weight.copy_(old_head.weight.detach().clone())
model.set_output_embeddings(new_head)
# Make sure the new head stays on the same device
model.get_output_embeddings().to(old_head.weight.device)

for p in model.parameters():
    p.requires_grad = False
for p in model.get_output_embeddings().parameters():
    p.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.4f}%)")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Trainable params: 136,134,656 / 630,167,424 (21.6029%)


## Train

In [6]:

out_dir = os.path.join(WORKDIR, "02_head_only_ckpt")

args = TrainingArguments(
    output_dir=out_dir,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=16,   # effective bs = 16
    learning_rate=2e-4,
    num_train_epochs=1,
    warmup_ratio=0.03,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=400,
    save_steps=400,
    save_total_limit=1,
    fp16=False,
    report_to=[],
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
)

trainer.train()


/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Step,Training Loss,Validation Loss


TrainOutput(global_step=125, training_loss=1.3093449556975158e+16, metrics={'train_runtime': 124.1338, 'train_samples_per_second': 16.112, 'train_steps_per_second': 1.007, 'total_flos': 2276502994944000.0, 'train_loss': 1.3093449556975158e+16, 'epoch': 1.0})

## Generate + ROUGE

In [7]:

def build_gen_prompt(article: str):
    inst = "Сделай краткое резюме новости."
    return f"[INST] {inst}\n\nТекст:\n{article}\n\nОтвет:\n[/INST]"

refs = test_df_s["summary"].tolist()
articles = test_df_s["text"].tolist()

model.eval()

def generate_one(article: str):
    prompt = build_gen_prompt(article)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(next(model.parameters()).device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            num_beams=2,
            early_stopping=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    txt = tokenizer.decode(out[0], skip_special_tokens=True)
    if "Ответ:" in txt:
        txt = txt.split("Ответ:", 1)[-1].strip()
    return txt.strip()

preds = []
for i, a in enumerate(articles):
    preds.append(generate_one(a))
    if (i+1) % 50 == 0:
        print("Generated", i+1, "/", len(articles))

scores = rouge_f1(preds, refs)
scores


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


Generated 50 / 400
Generated 100 / 400
Generated 150 / 400
Generated 200 / 400
Generated 250 / 400
Generated 300 / 400
Generated 350 / 400
Generated 400 / 400


{'rouge1': 0.1531566379937795,
 'rouge2': 0.049229604859551676,
 'rougeL': 0.14856914541753755}

## Save results

In [8]:

out = {
    "model": MODEL_NAME,
    "method": "head-only (untied lm_head)",
    "train_subset": int(TRAIN_N),
    "val_subset": int(VAL_N),
    "test_subset": int(TEST_N),
    "rouge_f1": scores,
}
out_path = os.path.join(WORKDIR, "02_head_only_rouge.json")
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(out, f, ensure_ascii=False, indent=2)
print("Saved:", out_path)


Saved: /kaggle/working/02_head_only_rouge.json
